# Ноутбук 02: GridSearchCV, финальный выбор и тест
Использует `model_feature_set_decisions.csv` из первого ноутбука.

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from lab_utils import *

# Загружаем данные и трансформируем (как в первом ноутбуке)
datasets = load_course_datasets()
splits = {}
for name, df in datasets.items():
    X, y = split_xy(df)
    X_train, X_valid, X_test, y_train, y_valid, y_test = train_valid_test_split_stratified(
        X, y, test_size=0.2, valid_size=0.2, random_state=SEED
    )
    splits[name] = (X_train, X_valid, X_test, y_train, y_valid, y_test)

transformed = {}
for name, (X_train, X_valid, X_test, y_train, y_valid, y_test) in splits.items():
    preprocessor = build_preprocessor(X_train)
    X_train_t, X_valid_t, X_test_t, feat_names = transform_with_names(
        preprocessor, X_train, X_valid, X_test
    )
    transformed[name] = (X_train_t, X_valid_t, X_test_t, feat_names, y_train, y_valid, y_test)

In [2]:
# Загружаем решения из первого ноутбука
decisions_path = OUTPUT_DIR / 'model_feature_set_decisions.csv'
if not decisions_path.exists():
    raise FileNotFoundError("Сначала выполните первый ноутбук!")
decisions = pd.read_csv(decisions_path)

# Для каждой пары dataset+model проведём GridSearchCV
all_gs_results = []
tuned_pipelines = {}

models = make_tuning_models()
param_grids = make_param_grids()

for ds_name in ['medical', 'finance']:
    for model_name in models.keys():
        row = decisions[(decisions['dataset']==ds_name) & (decisions['model']==model_name)]
        if row.empty:
            continue
        selected_fs = row.iloc[0]['selected_feature_set']
        # получаем список признаков
        if selected_fs != 'full':
            _, _, _, feat_names, _, _, _ = transformed[ds_name]
            ranking_df = pd.read_csv(OUTPUT_DIR / 'feature_ranking.csv')
            selected_features = get_feature_set_features(ds_name, selected_fs, feat_names, ranking_df)
        else:
            selected_features = None
        pipeline = build_model_pipeline(models[model_name], selected_features)
        X_train_t, _, _, _, y_train, _, _ = transformed[ds_name]
        
        gs = GridSearchCV(pipeline, param_grids[model_name], cv=3, scoring='f1', n_jobs=-1, verbose=0)
        gs.fit(X_train_t, y_train)
        tuned_pipelines[(ds_name, model_name)] = gs.best_estimator_
        
        cv_df = pd.DataFrame(gs.cv_results_)
        top5 = top_gridsearch_rows(cv_df, ds_name, selected_fs, model_name, top_n=5)
        all_gs_results.append(top5)

gs_top_df = pd.concat(all_gs_results, ignore_index=True)
gs_top_df.to_csv(OUTPUT_DIR / 'gridsearch_results_top.csv', index=False)
print("Saved gridsearch_results_top.csv")

FileNotFoundError: Сначала выполните первый ноутбук!

In [ ]:
# Сравнение baseline vs tuned на validation
val_rows = []
for ds_name in ['medical', 'finance']:
    for model_name in models.keys():
        row = decisions[(decisions['dataset']==ds_name) & (decisions['model']==model_name)]
        if row.empty:
            continue
        selected_fs = row.iloc[0]['selected_feature_set']
        if selected_fs != 'full':
            _, _, _, feat_names, _, _, _ = transformed[ds_name]
            ranking_df = pd.read_csv(OUTPUT_DIR / 'feature_ranking.csv')
            selected_features = get_feature_set_features(ds_name, selected_fs, feat_names, ranking_df)
        else:
            selected_features = None
        X_train_t, X_valid_t, _, _, y_train, y_valid, _ = transformed[ds_name]
        
        # baseline (default params)
        base_pipe = build_model_pipeline(models[model_name], selected_features)
        base_pipe.fit(X_train_t, y_train)
        base_metrics = evaluate_fitted_model(base_pipe, X_valid_t, y_valid)
        # tuned
        tuned_pipe = tuned_pipelines[(ds_name, model_name)]
        tuned_metrics = evaluate_fitted_model(tuned_pipe, X_valid_t, y_valid)
        
        val_rows.append({'dataset': ds_name, 'model': model_name, 'feature_set': selected_fs,
                         'variant': 'baseline_default',
                         'accuracy': base_metrics['accuracy'], 'f1': base_metrics['f1'], 'roc_auc': base_metrics['roc_auc']})
        val_rows.append({'dataset': ds_name, 'model': model_name, 'feature_set': selected_fs,
                         'variant': 'tuned_best',
                         'accuracy': tuned_metrics['accuracy'], 'f1': tuned_metrics['f1'], 'roc_auc': tuned_metrics['roc_auc']})

val_df = pd.DataFrame(val_rows)
print("Validation comparison (baseline vs tuned):")
print(val_df)

In [ ]:
# Финальный выбор победителя по validation f1
final_models = {}
for ds_name in ['medical', 'finance']:
    subset = val_df[val_df['dataset']==ds_name]
    winner = choose_validation_winner(subset, ds_name)
    final_models[ds_name] = winner
    print(f"{ds_name} winner: {winner['model']} ({winner['variant']}) with F1={winner['f1']:.4f}")

In [ ]:
# Финальная проверка на test (только для выбранных моделей)
test_rows = []
for ds_name in ['medical', 'finance']:
    winner = final_models[ds_name]
    model_name = winner['model']
    variant = winner['variant']
    selected_fs = winner['feature_set']
    
    X_train_t, _, X_test_t, feat_names, y_train, _, y_test = transformed[ds_name]
    if selected_fs != 'full':
        ranking_df = pd.read_csv(OUTPUT_DIR / 'feature_ranking.csv')
        selected_features = get_feature_set_features(ds_name, selected_fs, feat_names, ranking_df)
    else:
        selected_features = None

    # baseline (default)
    base_pipe = build_model_pipeline(make_tuning_models()[model_name], selected_features)
    base_pipe.fit(X_train_t, y_train)
    base_test = evaluate_fitted_model(base_pipe, X_test_t, y_test)
    
    # tuned (если выбран tuned, иначе baseline)
    if variant == 'tuned_best':
        tuned_pipe = tuned_pipelines[(ds_name, model_name)]
        tuned_test = evaluate_fitted_model(tuned_pipe, X_test_t, y_test)
    else:
        tuned_test = base_test

    test_rows.append({'dataset': ds_name, 'feature_set': selected_fs, 'model': model_name,
                      'variant': 'baseline_default',
                      'accuracy': base_test['accuracy'], 'f1': base_test['f1'], 'roc_auc': base_test['roc_auc'],
                      'fit_time_sec': 0.0})
    test_rows.append({'dataset': ds_name, 'feature_set': selected_fs, 'model': model_name,
                      'variant': 'tuned_best',
                      'accuracy': tuned_test['accuracy'], 'f1': tuned_test['f1'], 'roc_auc': tuned_test['roc_auc'],
                      'fit_time_sec': 0.0})

test_df = pd.DataFrame(test_rows)
test_df.to_csv(OUTPUT_DIR / 'baseline_vs_tuned_test_results.csv', index=False)
print("Saved baseline_vs_tuned_test_results.csv")
test_df

## Narrative
- GridSearchCV: ...
- Результаты на test: ...
- Выводы: ...